# MMLU-Pro XGBoost — No Category (Question Text + LLM Model Only)

Train on `data/mmlu_pro_full_enriched.csv` using **only** three raw columns:
`question`, `llm_model`, `error`.

All features are derived from scratch:
- 30 text features engineered from the raw question string
- 1 WOE-encoded feature for `llm_model`
- 1 WOE-encoded expected-answer-type feature inferred from the question

Pipeline: Stratified 70/15/15 split → WOE encoding → IV filtering → RFE (top 25) → Complex XGBoost (1000 trees, depth=7, lr=0.02, early stopping).

Enriched dataset saved to `data/mmlu_pro_question_features.csv`.

In [1]:
import json
import re
import string
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.feature_selection import RFE
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "mmlu_pro_full_enriched.csv"
OUTPUT_DATA_PATH = PROJECT_ROOT / "data" / "mmlu_pro_question_features.csv"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_NAME = "xgboost_mmlu_pro_no_category"

RANDOM_STATE = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15
IV_THRESHOLD = 0
RFE_N_FEATURES = 25
NUMERIC_IV_BINS = 5

N_ESTIMATORS = 1000
MAX_DEPTH = 7
LEARNING_RATE = 0.02
EARLY_STOPPING_ROUNDS = 50

## Load data (question, llm_model, error only)

In [2]:
df = pd.read_csv(DATA_PATH, usecols=["question", "llm_model", "error"])
df["target"] = (df["error"] == "error").astype(int)

print(f"Rows: {len(df):,}")
print(f"Models: {df['llm_model'].nunique()}")
print(df["target"].value_counts())
print()
print(df.groupby(["llm_model", "target"]).size().unstack(fill_value=0).head())
df.head(3)

Rows: 563,787
Models: 47
target
1    282844
0    280943
Name: count, dtype: int64

target                0     1
llm_model                    
DeepSeek-Coder-V2  6586  3765
Llama-2-13b-hf     2830  9202
Llama-2-70b-hf     4366  7666
Llama-2-7b-hf      2207  9825
Meta-Llama-3-70B   6258  5774


,question,llm_model,error,target
0,"What will be the number of lamps, each having ...",DeepSeek-Coder-V2,no_error,0
1,Stack is also known as\n\nA. FIFO memory\nB. F...,DeepSeek-Coder-V2,no_error,0
2,The errors mainly caused by human mistakes are...,DeepSeek-Coder-V2,no_error,0


## Feature engineering helpers

In [3]:
STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "in", "on", "at", "to", "for", "of",
    "with", "by", "from", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "could", "should",
    "may", "might", "shall", "can", "this", "that", "these", "those", "it", "its",
    "he", "she", "they", "we", "you", "i", "me", "him", "her", "us", "them",
    "my", "your", "his", "our", "their", "what", "which", "who", "when", "where",
    "why", "how", "all", "each", "every", "both", "few", "more", "most", "other",
    "some", "such", "no", "nor", "not", "only", "same", "so", "than", "too",
    "very", "just", "as", "if", "about", "above", "after", "before", "between",
    "into", "through", "during", "without", "within", "along", "following",
    "across", "behind", "beyond", "plus", "except", "up", "out", "around",
    "down", "off", "over", "under", "again", "then", "once", "here", "there",
    "any", "also", "although", "because", "since", "while", "whether", "given",
}

MATH_OPERATORS = set("+-*/=\u2265\u2264\u2211\u221a%\u222b\u2202\u2207\xd7\xf7")
CHOICE_RE = re.compile(r"^\s*[A-J][\.\)]\s", re.MULTILINE)
YEAR_RE = re.compile(r"\b(19[0-9]{2}|20[0-2][0-9])\b")
SCIENTIFIC_RE = re.compile(r"\d+\.?\d*[eE][+\-]?\d+|\xd7\s*10")
CODE_KEYWORDS = re.compile(r"`|def |class |import |for |while |function |=>|\{|\}")
SUBORDINATORS = re.compile(
    r"\b(that|which|because|when|while|although|unless|since|after|before|until|whether|if)\b",
    re.IGNORECASE,
)
LEADING_Q_RE = re.compile(
    r"^(why did|what caused|how did|when did|who caused|what led|what made|why was|why were|why is|why are)",
    re.IGNORECASE,
)


def split_stem_choices(text):
    """Return (stem_text, list_of_choice_lines) by splitting on A. B. C. ... lines."""
    lines = str(text).split("\n")
    stem_lines, choice_lines, in_choices = [], [], False
    for line in lines:
        if CHOICE_RE.match(line):
            in_choices = True
            choice_lines.append(line.strip())
        elif not in_choices:
            stem_lines.append(line)
    return "\n".join(stem_lines), choice_lines


def classify_answer_type(stem_lower):
    """Infer expected answer type from the question stem."""
    if re.search(
        r"\b(calculate|compute|find the value|what is the value|how many|how much|"
        r"what percentage|what fraction|what number|evaluate)\b",
        stem_lower,
    ):
        return "numeric"
    if re.search(
        r"\b(true or false|which statement is (true|false|correct|incorrect)|is it true)\b",
        stem_lower,
    ):
        return "true_false"
    if re.search(r"\b(formula|equation|chemical formula|expression|symbol)\b", stem_lower):
        return "formula"
    if re.search(
        r"^(who|which person|which author|which scientist|which country|which city|where\b)",
        stem_lower[:60],
    ):
        return "named_entity"
    if re.search(r"^(what is|what are|define|definition|describe|explain)", stem_lower[:60]):
        return "concept"
    return "other"

In [4]:
def engineer_features(frame):
    """Derive all 31 features purely from question text and llm_model."""
    out = frame.copy()
    texts = out["question"].astype(str)

    # ── Stem + choices ───────────────────────────────────────────────────────
    parsed = texts.apply(split_stem_choices)
    stems = parsed.apply(lambda x: x[0])
    choices_list = parsed.apply(lambda x: x[1])

    # ── Length & size ────────────────────────────────────────────────────────
    words_all = texts.str.lower().str.findall(r"\b\w+\b")
    word_counts = words_all.apply(len)
    unique_words = words_all.apply(lambda w: len(set(w)))

    out["q_word_count"] = word_counts
    out["q_char_count"] = texts.apply(lambda t: sum(1 for c in t if not c.isspace()))
    out["q_newline_count"] = texts.str.count(r"\n")
    out["q_unique_word_count"] = unique_words
    out["q_avg_word_length"] = words_all.apply(
        lambda w: float(np.mean([len(x) for x in w])) if w else 0.0
    )
    out["q_type_token_ratio"] = (unique_words / word_counts.replace(0, np.nan)).fillna(0.0)

    sentences = texts.apply(
        lambda t: [s.strip() for s in re.split(r"[.?!]", t) if s.strip()]
    )
    sentence_counts = sentences.apply(len).clip(lower=1)
    out["q_sentence_count"] = sentence_counts
    out["q_avg_sentence_length"] = word_counts / sentence_counts

    # ── Answer-choice structure ──────────────────────────────────────────────
    out["q_num_choices"] = choices_list.apply(len)
    stem_words = stems.str.lower().str.findall(r"\b\w+\b")
    out["q_stem_word_count"] = stem_words.apply(len)
    out["q_stem_char_count"] = stems.apply(lambda t: sum(1 for c in t if not c.isspace()))
    choice_lengths = choices_list.apply(lambda c: [len(x) for x in c])
    out["q_avg_choice_length_chars"] = choice_lengths.apply(
        lambda l: float(np.mean(l)) if l else 0.0
    )

    # ── Numeric & math content ───────────────────────────────────────────────
    out["q_has_numbers"] = texts.str.contains(r"\d", regex=True).astype(int)
    out["q_digit_ratio"] = texts.apply(
        lambda t: sum(c.isdigit() for c in t) / max(len(t), 1)
    )
    out["q_has_math_operators"] = texts.apply(
        lambda t: int(any(c in MATH_OPERATORS for c in t))
    )
    out["q_math_operator_count"] = texts.apply(
        lambda t: sum(c in MATH_OPERATORS for c in t)
    )
    out["q_bracket_count"] = texts.apply(
        lambda t: sum(c in "()[]{}\u27e8\u27e9" for c in t)
    )
    out["q_has_scientific_notation"] = texts.apply(
        lambda t: int(bool(SCIENTIFIC_RE.search(t)))
    )

    # ── Text style ───────────────────────────────────────────────────────────
    out["q_uppercase_ratio"] = texts.apply(
        lambda t: sum(c.isupper() for c in t) / max(sum(c.isalpha() for c in t), 1)
    )
    out["q_punctuation_count"] = texts.apply(
        lambda t: sum(c in string.punctuation for c in t)
    )
    out["q_comma_count"] = texts.str.count(",")
    out["q_question_mark_count"] = texts.str.count(r"\?")
    out["q_long_word_ratio"] = words_all.apply(
        lambda w: sum(len(x) > 6 for x in w) / max(len(w), 1)
    )

    # ── Semantic flags ───────────────────────────────────────────────────────
    texts_lower = texts.str.lower()
    out["q_has_negation"] = texts_lower.str.contains(
        r"\b(not|except|never|cannot|n't)\b", regex=True
    ).astype(int)
    out["q_has_none_all_above"] = texts_lower.str.contains(
        r"none of the above|all of the above", regex=True
    ).astype(int)
    out["q_is_which_following"] = texts_lower.str.contains(
        r"which of the following", regex=True
    ).astype(int)
    out["q_has_code_pattern"] = texts.apply(
        lambda t: int(bool(CODE_KEYWORDS.search(t)))
    )

    stems_lower = stems.str.lower().str.strip()
    out["q_starts_with_what"] = stems_lower.str.startswith("what").astype(int)
    out["q_starts_with_how"] = stems_lower.str.startswith("how").astype(int)
    out["q_starts_with_why"] = stems_lower.str.startswith("why").astype(int)
    out["q_starts_with_which"] = stems_lower.str.startswith("which").astype(int)

    # ── Density ──────────────────────────────────────────────────────────────
    out["q_stopword_ratio"] = words_all.apply(
        lambda w: sum(x in STOPWORDS for x in w) / max(len(w), 1)
    )

    # ── Temporal / recency ───────────────────────────────────────────────────
    years_found = texts.apply(lambda t: [int(y) for y in YEAR_RE.findall(t)])
    out["q_has_year_mention"] = years_found.apply(lambda y: int(len(y) > 0))
    out["q_max_year_mentioned"] = years_found.apply(lambda y: max(y) if y else 0)
    # Recency gap: how far back the question's most recent year is from 2024
    out["q_recency_gap"] = out["q_max_year_mentioned"].apply(
        lambda y: 2024 - y if y > 0 else 0
    )

    # ── Named-entity proxies ─────────────────────────────────────────────────
    def count_named_entities(text):
        sentence_starts = {
            s.strip().split()[0]
            for s in re.split(r"[.?!\n]", text)
            if s.strip() and s.strip().split()
        }
        tokens = re.findall(r"\b([A-Z][a-z]+)\b", text)
        return sum(1 for t in tokens if t not in sentence_starts)

    out["q_num_named_entities"] = texts.apply(count_named_entities)
    out["q_has_rare_entity"] = texts.apply(
        lambda t: int(bool(re.search(r"\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)+\b", t)))
    )

    # ── Presupposition / leading-question proxy ──────────────────────────────
    out["q_is_leading_question"] = stems_lower.apply(
        lambda s: int(bool(LEADING_Q_RE.match(s)))
    )

    # ── Syntactic-depth proxy (subordinating conjunction count) ──────────────
    out["q_dependency_depth_proxy"] = texts_lower.apply(
        lambda t: len(SUBORDINATORS.findall(t))
    )

    # ── Expected answer type (categorical — WOE-encoded downstream) ──────────
    out["q_expected_answer_type"] = stems_lower.apply(classify_answer_type)

    return out

In [5]:
print("Engineering features — this takes a few minutes on 560k rows...")
df = engineer_features(df)
print(f"Done. Shape: {df.shape}")

print(f"\nq_expected_answer_type distribution:")
print(df["q_expected_answer_type"].value_counts())

MODEL_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_DATA_PATH, index=False)
print(f"\nSaved enriched dataset → {OUTPUT_DATA_PATH}")
df.head(3)

Engineering features — this takes a few minutes on 560k rows...


/var/folders/bp/p_1nzxj96b7gxxrykwzbq04r0000gn/T/ipykernel_64409/885082771.py:75: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  out["q_has_negation"] = texts_lower.str.contains(


Done. Shape: (563787, 44)

q_expected_answer_type distribution:
q_expected_answer_type
other           435425
numeric          81956
concept          33969
formula          10412
named_entity      1696
true_false         329
Name: count, dtype: int64

Saved enriched dataset → /Users/konstantine25b/Desktop/Gaia Student Club/Retrival Failure/data/mmlu_pro_question_features.csv


,question,llm_model,error,target,q_word_count,q_char_count,q_newline_count,q_unique_word_count,q_avg_word_length,q_type_token_ratio,...,q_starts_with_which,q_stopword_ratio,q_has_year_mention,q_max_year_mentioned,q_recency_gap,q_num_named_entities,q_has_rare_entity,q_is_leading_question,q_dependency_depth_proxy,q_expected_answer_type
0,"What will be the number of lamps, each having ...",DeepSeek-Coder-V2,no_error,0,46,145,11,44,2.847826,0.956522,...,0,0.282609,0,0,0,0,0,0,0,other
1,Stack is also known as\n\nA. FIFO memory\nB. F...,DeepSeek-Coder-V2,no_error,0,17,67,5,14,3.705882,0.823529,...,0,0.235294,0,0,0,0,0,0,0,other
2,The errors mainly caused by human mistakes are...,DeepSeek-Coder-V2,no_error,0,17,90,4,15,4.941176,0.882353,...,0,0.235294,0,0,0,0,0,0,0,other


## Train / validation / test split

In [6]:
train_val_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["target"],
)

val_ratio = VAL_SIZE / (1 - TEST_SIZE)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=val_ratio,
    random_state=RANDOM_STATE,
    stratify=train_val_df["target"],
)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(train_df["target"].value_counts(normalize=True).round(3))

Train: 394,650 | Val: 84,568 | Test: 84,569
target
1    0.502
0    0.498
Name: proportion, dtype: float64


## WOE encoding and feature matrix

In [7]:
# Categorical columns that get WOE-encoded
CATEGORICAL_COLUMNS = ["llm_model", "q_expected_answer_type"]

# Binary / boolean features (already 0/1)
BOOLEAN_COLUMNS = [
    "q_has_numbers",
    "q_has_math_operators",
    "q_has_scientific_notation",
    "q_has_code_pattern",
    "q_has_negation",
    "q_has_none_all_above",
    "q_is_which_following",
    "q_starts_with_what",
    "q_starts_with_how",
    "q_starts_with_why",
    "q_starts_with_which",
    "q_has_year_mention",
    "q_has_rare_entity",
    "q_is_leading_question",
]

# Continuous numeric features
NUMERIC_COLUMNS = [
    "q_word_count",
    "q_char_count",
    "q_sentence_count",
    "q_newline_count",
    "q_unique_word_count",
    "q_avg_word_length",
    "q_type_token_ratio",
    "q_avg_sentence_length",
    "q_num_choices",
    "q_stem_word_count",
    "q_stem_char_count",
    "q_avg_choice_length_chars",
    "q_digit_ratio",
    "q_math_operator_count",
    "q_bracket_count",
    "q_uppercase_ratio",
    "q_punctuation_count",
    "q_comma_count",
    "q_question_mark_count",
    "q_long_word_ratio",
    "q_stopword_ratio",
    "q_max_year_mentioned",
    "q_recency_gap",
    "q_num_named_entities",
    "q_dependency_depth_proxy",
]


def compute_woe_maps(frame, columns, target="target"):
    maps = {}
    total_events = frame[target].sum()
    total_non_events = len(frame) - total_events
    for column in columns:
        grouped = frame.groupby(column, dropna=False)[target].agg(["sum", "count"])
        grouped["non_events"] = grouped["count"] - grouped["sum"]
        woe_map = {}
        for value, row in grouped.iterrows():
            event_rate = (row["sum"] + 0.5) / (total_events + 1.0)
            non_event_rate = (row["non_events"] + 0.5) / (total_non_events + 1.0)
            woe_map[value] = float(np.log(event_rate / non_event_rate))
        maps[column] = woe_map
    return maps


def apply_woe(frame, columns, maps):
    transformed = frame.copy()
    for column in columns:
        transformed[f"{column}_woe"] = transformed[column].map(maps[column]).fillna(0.0)
    return transformed


def build_feature_matrix(frame):
    numeric = frame[NUMERIC_COLUMNS].apply(pd.to_numeric, errors="coerce")
    boolean = frame[BOOLEAN_COLUMNS].astype(int)
    woe_cols = [f"{col}_woe" for col in CATEGORICAL_COLUMNS]
    return pd.concat([numeric, boolean, frame[woe_cols]], axis=1)


woe_maps = compute_woe_maps(train_df, CATEGORICAL_COLUMNS)
train_df = apply_woe(train_df, CATEGORICAL_COLUMNS, woe_maps)
val_df = apply_woe(val_df, CATEGORICAL_COLUMNS, woe_maps)
test_df = apply_woe(test_df, CATEGORICAL_COLUMNS, woe_maps)

feature_names = NUMERIC_COLUMNS + BOOLEAN_COLUMNS + [f"{col}_woe" for col in CATEGORICAL_COLUMNS]

x_train = build_feature_matrix(train_df)
x_val = build_feature_matrix(val_df)
x_test = build_feature_matrix(test_df)

medians = x_train.median(numeric_only=True)
x_train = x_train.fillna(medians)
x_val = x_val.fillna(medians)
x_test = x_test.fillna(medians)

y_train = train_df["target"]
y_val = val_df["target"]
y_test = test_df["target"]

print(f"Feature count: {len(feature_names)}")
x_train.head()

Feature count: 41


,q_word_count,q_char_count,q_sentence_count,q_newline_count,q_unique_word_count,q_avg_word_length,q_type_token_ratio,q_avg_sentence_length,q_num_choices,q_stem_word_count,...,q_is_which_following,q_starts_with_what,q_starts_with_how,q_starts_with_why,q_starts_with_which,q_has_year_mention,q_has_rare_entity,q_is_leading_question,llm_model_woe,q_expected_answer_type_woe
436604,47,145,13,11,41,2.659574,0.872340,3.615385,10,27,...,0,0,0,0,0,1,0,0,-0.832226,0.106791
477931,211,1065,17,16,139,4.890995,0.658768,12.411765,10,176,...,1,0,0,0,0,0,1,0,-0.532696,-0.014420
500951,58,352,13,11,49,5.689655,0.844828,4.461538,10,28,...,1,0,0,0,0,0,1,0,-1.479141,-0.014420
461057,62,189,16,11,44,2.790323,0.709677,3.875000,10,42,...,0,0,0,0,0,0,0,0,-1.100755,0.337809
319630,131,669,13,11,86,4.961832,0.656489,10.076923,10,25,...,0,0,0,0,0,0,0,0,-1.577836,-0.014420


## Information Value (IV) filtering

In [8]:
def compute_iv_grouped(frame, group_col, target="target", observed=False):
    events = frame[target].sum()
    non_events = len(frame) - events
    if events == 0 or non_events == 0:
        return 0.0
    grouped = frame.groupby(group_col, dropna=False, observed=observed)[target].agg(
        ["sum", "count"]
    )
    iv = 0.0
    for _, row in grouped.iterrows():
        bad_dist = row["sum"] / events
        good_dist = (row["count"] - row["sum"]) / non_events
        if bad_dist <= 0 or good_dist <= 0:
            continue
        woe = np.log(bad_dist / good_dist)
        iv += (bad_dist - good_dist) * woe
    return float(iv)


def compute_feature_iv(train_frame, feature_name, medians):
    if feature_name.endswith("_woe"):
        raw_col = feature_name.removesuffix("_woe")
        return compute_iv_grouped(train_frame, raw_col)
    if feature_name in BOOLEAN_COLUMNS:
        return compute_iv_grouped(train_frame, feature_name)
    filled = train_frame[feature_name].fillna(
        medians.get(feature_name, train_frame[feature_name].median())
    )
    try:
        binned = pd.qcut(filled, q=NUMERIC_IV_BINS, duplicates="drop")
    except ValueError:
        return 0.0
    temp = pd.DataFrame({"bin": binned, "target": train_frame["target"]})
    return compute_iv_grouped(temp, "bin", observed=True)


iv_scores = {
    name: compute_feature_iv(train_df, name, medians) for name in feature_names
}
iv_df = pd.Series(iv_scores, name="iv").sort_values(ascending=False).reset_index()
iv_df.columns = ["feature", "iv"]
iv_df["passes_threshold"] = iv_df["iv"] >= IV_THRESHOLD
iv_df

,feature,iv,passes_threshold
0,llm_model_woe,0.566920,True
1,q_stem_word_count,0.143079,True
2,q_stem_char_count,0.123633,True
3,q_punctuation_count,0.110890,True
4,q_sentence_count,0.076561,True
5,q_has_numbers,0.069272,True
6,q_digit_ratio,0.066600,True
7,q_avg_word_length,0.060772,True
8,q_long_word_ratio,0.054091,True
9,q_math_operator_count,0.045320,True


In [9]:
iv_features = iv_df.loc[iv_df["passes_threshold"], "feature"].tolist()
if not iv_features:
    iv_features = [iv_df.iloc[0]["feature"]]
print(f"IV selected {len(iv_features)} / {len(feature_names)} features (threshold={IV_THRESHOLD})")
iv_features

IV selected 41 / 41 features (threshold=0)


['llm_model_woe',
 'q_stem_word_count',
 'q_stem_char_count',
 'q_punctuation_count',
 'q_sentence_count',
 'q_has_numbers',
 'q_digit_ratio',
 'q_avg_word_length',
 'q_long_word_ratio',
 'q_math_operator_count',
 'q_expected_answer_type_woe',
 'q_bracket_count',
 'q_word_count',
 'q_has_math_operators',
 'q_starts_with_what',
 'q_comma_count',
 'q_stopword_ratio',
 'q_unique_word_count',
 'q_type_token_ratio',
 'q_has_scientific_notation',
 'q_char_count',
 'q_avg_choice_length_chars',
 'q_has_code_pattern',
 'q_num_named_entities',
 'q_uppercase_ratio',
 'q_starts_with_why',
 'q_newline_count',
 'q_starts_with_how',
 'q_avg_sentence_length',
 'q_is_leading_question',
 'q_dependency_depth_proxy',
 'q_has_negation',
 'q_starts_with_which',
 'q_has_year_mention',
 'q_recency_gap',
 'q_is_which_following',
 'q_num_choices',
 'q_has_rare_entity',
 'q_has_none_all_above',
 'q_question_mark_count',
 'q_max_year_mentioned']

## Recursive Feature Elimination (RFE — top 25)

In [10]:
rfe_estimator = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rfe = RFE(estimator=rfe_estimator, n_features_to_select=RFE_N_FEATURES, step=1)
rfe.fit(x_train[iv_features], y_train)

rfe_df = pd.DataFrame(
    {"feature": iv_features, "ranking": rfe.ranking_, "selected": rfe.support_}
).sort_values(["selected", "ranking"], ascending=[False, True]).reset_index(drop=True)

selected_features = rfe_df.loc[rfe_df["selected"], "feature"].tolist()
print(f"RFE selected {len(selected_features)} features")
rfe_df

RFE selected 25 features


,feature,ranking,selected
0,llm_model_woe,1,True
1,q_stem_word_count,1,True
2,q_stem_char_count,1,True
3,q_punctuation_count,1,True
4,q_sentence_count,1,True
5,q_digit_ratio,1,True
6,q_avg_word_length,1,True
7,q_long_word_ratio,1,True
8,q_math_operator_count,1,True
9,q_expected_answer_type_woe,1,True


## XGBoost training

In [11]:
model = xgb.XGBClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model.fit(
    x_train[selected_features],
    y_train,
    eval_set=[(x_val[selected_features], y_val)],
    verbose=100,
)

print(f"\nBest iteration: {model.best_iteration}")

[0]	validation_0-logloss:0.68944
[100]	validation_0-logloss:0.58019
[200]	validation_0-logloss:0.56168
[300]	validation_0-logloss:0.54926
[400]	validation_0-logloss:0.53852
[500]	validation_0-logloss:0.52925
[600]	validation_0-logloss:0.52097
[700]	validation_0-logloss:0.51348
[800]	validation_0-logloss:0.50655
[900]	validation_0-logloss:0.50049
[999]	validation_0-logloss:0.49445

Best iteration: 999


## Evaluation

In [12]:
y_pred = model.predict(x_test[selected_features])
y_proba = model.predict_proba(x_test[selected_features])[:, 1]

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f"Accuracy : {acc:.4f}")
print(f"F1       : {f1:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"ROC-AUC  : {auc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["no_error", "error"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.7663
F1       : 0.7670
Precision: 0.7674
Recall   : 0.7666
ROC-AUC  : 0.8457

              precision    recall  f1-score   support

    no_error       0.77      0.77      0.77     42142
       error       0.77      0.77      0.77     42427

    accuracy                           0.77     84569
   macro avg       0.77      0.77      0.77     84569
weighted avg       0.77      0.77      0.77     84569

Confusion matrix:
[[32281  9861]
 [ 9901 32526]]


In [13]:
model.save_model(str(MODEL_DIR / f"{MODEL_NAME}.json"))

artifacts = {
    "model_name": MODEL_NAME,
    "selected_features": selected_features,
    "woe_maps": {
        k: {str(kk): vv for kk, vv in v.items()} for k, v in woe_maps.items()
    },
    "medians": medians.to_dict(),
    "metrics": {
        "accuracy": round(float(acc), 4),
        "f1": round(float(f1), 4),
        "precision": round(float(prec), 4),
        "recall": round(float(rec), 4),
        "roc_auc": round(float(auc), 4),
    },
}

artifact_path = MODEL_DIR / f"{MODEL_NAME}_artifacts.json"
with open(artifact_path, "w") as fh:
    json.dump(artifacts, fh, indent=2)

print(f"Model     → {MODEL_DIR / f'{MODEL_NAME}.json'}")
print(f"Artifacts → {artifact_path}")
print(f"\nFinal selected features ({len(selected_features)}):")
for feat in selected_features:
    print(f"  {feat}")

Model     → /Users/konstantine25b/Desktop/Gaia Student Club/Retrival Failure/models/xgboost_mmlu_pro_no_category.json
Artifacts → /Users/konstantine25b/Desktop/Gaia Student Club/Retrival Failure/models/xgboost_mmlu_pro_no_category_artifacts.json

Final selected features (25):
  llm_model_woe
  q_stem_word_count
  q_stem_char_count
  q_punctuation_count
  q_sentence_count
  q_digit_ratio
  q_avg_word_length
  q_long_word_ratio
  q_math_operator_count
  q_expected_answer_type_woe
  q_bracket_count
  q_word_count
  q_starts_with_what
  q_stopword_ratio
  q_unique_word_count
  q_type_token_ratio
  q_char_count
  q_avg_choice_length_chars
  q_newline_count
  q_has_negation
  q_starts_with_which
  q_is_which_following
  q_num_choices
  q_has_rare_entity
  q_question_mark_count
